# Day 2 — Computer Vision & Perception for UR5

Welcome to Day 2 of the **UR5 AI Workshop**!

### 🎯 Learning Objectives
By the end of this notebook, you will:
1. Capture and process images from a camera.
2. Apply computer vision algorithms (edges, contours, color filtering).
3. Use pre-trained AI models for object detection.
4. Calibrate the camera against the robot workspace with a checkerboard.
5. Map image coordinates to robot coordinates and guide the UR5 in URSim.

### 🧰 Before you start
- A webcam looking down at the table, which must **not move** after calibration (Exercise 5).
- A printed checkerboard with **10 × 7 squares** (9 × 6 inner corners), for example from the OpenCV pattern generator. Measure the square size on the printout.
- A few objects of different colors (red, blue, green, yellow).
- URSim running (`ursim/bringup.sh`) for the Final Challenge.

**Image windows:** live video opens an OpenCV window. Press **q** or **Esc**, or close the window, to stop. Still images are shown inline in the notebook.

### 🎓 Instructor version
This notebook contains the reference solution of the Final Challenge. Do not distribute it before the session.

In [ ]:
# Setup — run this cell first
import glob
import math
import os
import time

# OpenCV's bundled Qt only ships the X11 plugin. On a Wayland desktop this avoids the
# 'Could not find the Qt platform plugin "wayland"' warning (windows go through XWayland).
os.environ.setdefault('QT_QPA_PLATFORM', 'xcb')
import cv2
import matplotlib.pyplot as plt
import numpy as np

CAMERA_INDEX = 0   # change if you have several cameras

def show(img, title='', size=(8, 6)):
    """Display a BGR (color) or single-channel image inline."""
    plt.figure(figsize=size)
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

def run_live(window, process=None, on_key=None, timeout=120):
    """Show the live camera feed in an OpenCV window and return the last raw frame.

    process(frame) -> image to display (optional)
    on_key(key, frame) is called for every key press (optional)
    Stops on q / Esc, when the window is closed, or after `timeout` seconds.
    """
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera {CAMERA_INDEX}. Check the cable, or try another CAMERA_INDEX.')
    last, t0 = None, time.time()
    try:
        while time.time() - t0 < timeout:
            ok, frame = cap.read()
            if not ok:
                print('Camera read failed')
                break
            last = frame
            cv2.imshow(window, process(frame.copy()) if process else frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27) or cv2.getWindowProperty(window, cv2.WND_PROP_VISIBLE) < 1:
                break
            if on_key and key != 255:
                on_key(key, frame)
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)   # let the window actually close
    return last

def load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f'{path} not found. Run the exercise that creates it first.')
    return img

## 🧩 Exercise 1 — Capture and Display Images
Open the live feed, place your objects in view, then press **q** (or close the window). The last frame is saved as `snapshot.jpg` and used in the next exercises.

In [ ]:
# Student Task:
frame = run_live('UR5 Camera Feed')
if frame is None:
    print('No frame captured')
else:
    cv2.imwrite('snapshot.jpg', frame)
    print('Saved snapshot.jpg', frame.shape)
    show(frame, 'snapshot.jpg')

## 🧩 Exercise 2 — Basic Image Processing (Edges & Contours)
A light blur before `Canny` removes sensor noise, which would otherwise create many tiny false edges.

In [ ]:
# Student Task:
img = load_image('snapshot.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 100, 200)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

result = img.copy()
cv2.drawContours(result, contours, -1, (0, 255, 0), 2)
print(f'{len(contours)} contours found')
show(edges, 'Canny edges')
show(result, 'Contours')

## 🧩 Exercise 3 — Color-Based Object Detection (HSV)
Pipeline (slide *Chaîne de traitement visuel*): **BGR → HSV → color mask → contours → features** (centroid, area, orientation).

HSV separates the color (*hue*) from the brightness (*value*), so a range on H stays valid when the lighting changes. In OpenCV, **H goes from 0 to 179**. Red sits at both ends of that scale, so it needs **two** ranges.

In [ ]:
# Student Task:
COLOR_RANGES = {   # name: list of (lower HSV, upper HSV)
    'red':    [((0, 120, 70), (10, 255, 255)), ((170, 120, 70), (179, 255, 255))],
    'yellow': [((20, 100, 100), (35, 255, 255))],
    'green':  [((40, 80, 50), (85, 255, 255))],
    'blue':   [((100, 150, 50), (130, 255, 255))],
}
DRAW_COLORS = {'red': (0, 0, 255), 'yellow': (0, 220, 220), 'green': (0, 200, 0), 'blue': (255, 0, 0)}
MIN_AREA = 500     # px², ignore smaller blobs (noise)

def color_mask(hsv, ranges):
    mask = np.zeros(hsv.shape[:2], np.uint8)
    for lower, upper in ranges:
        mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)    # remove small specks
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)   # fill small holes
    return mask

def detect_colored_objects(img, colors=COLOR_RANGES, min_area=MIN_AREA):
    """Return a list of objects: color, center (px), area (px²), angle (deg), bbox, sorted by area."""
    hsv = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_BGR2HSV)
    objects = []
    for name, ranges in colors.items():
        contours, _ = cv2.findContours(color_mask(hsv, ranges), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            area = cv2.contourArea(c)
            if area < min_area:
                continue
            m = cv2.moments(c)
            center = (m['m10'] / m['m00'], m['m01'] / m['m00'])
            angle = cv2.minAreaRect(c)[2]
            objects.append(dict(color=name, center=center, area=area, angle=angle, bbox=cv2.boundingRect(c)))
    return sorted(objects, key=lambda o: -o['area'])

def draw_objects(img, objects):
    for o in objects:
        x, y, w, h = o['bbox']
        cx, cy = map(int, o['center'])
        col = DRAW_COLORS.get(o['color'], (255, 255, 255))
        cv2.rectangle(img, (x, y), (x + w, y + h), col, 2)
        cv2.circle(img, (cx, cy), 5, col, -1)
        cv2.putText(img, f"{o['color']} ({cx},{cy})", (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    return img

img = load_image('snapshot.jpg')
objects = detect_colored_objects(img)
for o in objects:
    print(f"{o['color']:6s} center=({o['center'][0]:6.1f}, {o['center'][1]:6.1f}) px  "
          f"area={o['area']:7.0f} px²  angle={o['angle']:5.1f}°")
show(draw_objects(img.copy(), objects), 'Color detection')

**Tuning the ranges:** if an object is missed, read its HSV value and adjust `COLOR_RANGES`. Print the mask of one color to see what is kept.

In [ ]:
# Student Task:
def hsv_at(img, x, y):
    """HSV value of pixel (x, y), useful to tune COLOR_RANGES."""
    return cv2.cvtColor(img[y:y + 1, x:x + 1], cv2.COLOR_BGR2HSV)[0, 0]

if objects:
    cx, cy = map(int, objects[0]['center'])
    print(f"HSV at the center of the largest object ({objects[0]['color']}):", hsv_at(img, cx, cy))

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
show(color_mask(hsv, COLOR_RANGES['blue']), 'Mask: blue')

**Live tracking:** run the same detector on the video stream. Test its robustness by changing the lighting (lamp on/off, shadows, window light) and moving several objects of different colors.

In [ ]:
# Student Task:
run_live('Color tracking', process=lambda f: draw_objects(f, detect_colored_objects(f)))

## 🧩 Exercise 4 — Object Detection using a Pre-trained Model
MobileNet-SSD is a neural network trained on the **PASCAL VOC** dataset. It only knows the 20 classes below (bottle, chair, person…), so it will **not** recognise cubes or typical workpieces. Color detection (Exercise 3) is what we use for picking. Try it on a bottle or a person.

The model files `MobileNetSSD_deploy.prototxt` and `MobileNetSSD_deploy.caffemodel` must be in the project root, one level above this notebook (run `scripts/download_models.sh`). This exercise needs OpenCV 4.x, because OpenCV 5 removed `readNetFromCaffe`.

In [ ]:
# Student Task:
CLASSES = ['background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair',
           'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']

net = cv2.dnn.readNetFromCaffe('../MobileNetSSD_deploy.prototxt', '../MobileNetSSD_deploy.caffemodel')

def detect_with_mobilenet(img, min_confidence=0.4):
    h, w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(img, (300, 300)), 0.007843, (300, 300), 127.5)
    net.setInput(blob)
    detections = net.forward()
    results = []
    for i in range(detections.shape[2]):
        confidence = float(detections[0, 0, i, 2])
        if confidence < min_confidence:
            continue
        idx = int(detections[0, 0, i, 1])
        box = np.clip(detections[0, 0, i, 3:7], 0, 1) * np.array([w, h, w, h])
        results.append((CLASSES[idx], confidence, box.astype(int)))
    return results

img = load_image('snapshot.jpg')
results = detect_with_mobilenet(img)
for label, confidence, (x1, y1, x2, y2) in results:
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img, f'{label}: {confidence * 100:.1f}%', (x1, max(y1 - 10, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
print(results if results else 'No known object detected (only the 20 VOC classes above)')
show(img, 'MobileNet-SSD detections')

## 🧩 Exercise 5 — Camera Calibration with a Checkerboard
To send the robot to an object seen by the camera, we need the transformation **pixel (u, v) → robot (X, Y) on the table plane**. For a flat plane this is a **homography**: a 3×3 matrix $H$ with

$$\begin{pmatrix} X \\ Y \\ 1 \end{pmatrix} \sim H \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}$$

It handles scale, rotation, offset **and** the perspective of a camera that is not perfectly vertical, unlike a simple scale + offset.

**Steps:**
1. Lay the checkerboard flat on the table, in the robot workspace, and measure its square size.
2. Run the preview below. It labels three inner corners: **A** (closest to the image's top-left), **B** (the other end of A's row) and **C** (the other end of A's column).
3. Measure the robot (X, Y) of **A**, **B** and **C**. On a real robot, jog the TCP onto each corner and read `rtde_r.getActualTCPPose()`: this is the classic 3-point *touch-up*, also used by Polyscope's *Plane* feature. In this workshop, use the values your instructor gives you.
4. Take several snapshots (press **c**) without moving the camera or the board. Several shots average out detection noise.
5. Compute $H$ with `cv2.findHomography` and check the reprojection error.

**Why three measured corners?** The detector numbers the corners from either end of the board, and a camera looking down sees the table **mirrored** relative to the robot frame. A wrong orientation still gives a perfect fit, with the robot sent to the wrong place! The code fixes the order (A = top-left in the image), and three measured points give the real rotation *and* mirroring.

⚠️ After calibration the camera must not move. The homography is only exact for points **on the table plane**; the top of a tall object is slightly shifted (parallax).

In [ ]:
# Student Task:
BOARD_INNER = (9, 6)              # inner corners per row, per column — a 10 × 7 squares board
SQUARE_SIZE = 0.025               # m, measure it on YOUR printout (used to check A, B, C)

# Robot (X, Y) in metres of the corners labelled A, B, C in the preview
CORNER_A = (-0.55, -0.35)
CORNER_B = (-0.35, -0.35)         # 8 squares from A along the row
CORNER_C = (-0.55, -0.225)        # 5 squares from A along the column

def find_board(img):
    """Return the (N, 2) inner-corner pixels, always ordered from corner A (image top-left), or None."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    found, corners = cv2.findChessboardCornersSB(gray, BOARD_INNER)
    if not found:
        return None
    cols, rows = BOARD_INNER
    grid = corners.reshape(rows, cols, 2)
    # The detector may start from any end of the board: flip so that A is the top-left-most corner
    options = [grid, grid[::-1], grid[:, ::-1], grid[::-1, ::-1]]
    grid = min(options, key=lambda g: g[0, 0].sum())
    return grid.reshape(-1, 2)

def draw_board(img, corners):
    cols, rows = BOARD_INNER
    cv2.drawChessboardCorners(img, BOARD_INNER, corners.reshape(-1, 1, 2).astype(np.float32), True)
    for label, idx in (('A', 0), ('B', cols - 1), ('C', (rows - 1) * cols)):
        x, y = map(int, corners[idx])
        cv2.circle(img, (x, y), 12, (0, 0, 255), 3)
        cv2.putText(img, label, (x + 12, y - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 3)
    return img

def board_points_robot():
    """Robot (X, Y) of every inner corner, in the same order as find_board()."""
    cols, rows = BOARD_INNER
    a, b, c = (np.array(p, dtype=float) for p in (CORNER_A, CORNER_B, CORNER_C))
    step_row, step_col = (b - a) / (cols - 1), (c - a) / (rows - 1)    # one square, in robot coordinates
    i, j = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2).T                  # column / row index of each corner
    return a + np.outer(i, step_row) + np.outer(j, step_col)

def check_measurements(tolerance=0.005):
    """Compare the measured A, B, C with the printed square size."""
    cols, rows = BOARD_INNER
    a, b, c = (np.array(p, dtype=float) for p in (CORNER_A, CORNER_B, CORNER_C))
    ab, ac = np.linalg.norm(b - a), np.linalg.norm(c - a)
    angle = math.degrees(math.acos(np.dot(b - a, c - a) / (ab * ac)))
    print(f'|AB| = {ab * 1000:.1f} mm (expected {(cols - 1) * SQUARE_SIZE * 1000:.1f}), '
          f'|AC| = {ac * 1000:.1f} mm (expected {(rows - 1) * SQUARE_SIZE * 1000:.1f}), angle A = {angle:.1f}°')
    ok = (abs(ab - (cols - 1) * SQUARE_SIZE) < tolerance and abs(ac - (rows - 1) * SQUARE_SIZE) < tolerance
          and abs(angle - 90) < 3)
    print('✅ measurements consistent' if ok else '⚠️ check CORNER_A/B/C and SQUARE_SIZE')
    return ok

def preview(frame):
    corners = find_board(frame)
    if corners is not None:
        draw_board(frame, corners)
    shots = len(glob.glob('calib_*.jpg'))
    cv2.putText(frame, f'c = capture ({shots} saved), q = done', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return frame

def capture(key, frame):
    if key == ord('c'):
        if find_board(frame) is None:
            print('Board not found — not saved')
            return
        path = f'calib_{len(glob.glob("calib_*.jpg")):02d}.jpg'
        cv2.imwrite(path, frame)
        print('Saved', path)

check_measurements()
run_live('Calibration — press c to capture', process=preview, on_key=capture)

In [ ]:
# Student Task:
pixels, world = [], []
for path in sorted(glob.glob('calib_*.jpg')):
    corners = find_board(load_image(path))
    if corners is None:
        print(f'{path}: board not found, skipped')
        continue
    pixels.append(corners)
    world.append(board_points_robot())
if not pixels:
    raise RuntimeError('No usable calibration image. Capture some with the cell above.')

pixels = np.vstack(pixels).astype(np.float64)
world = np.vstack(world).astype(np.float64)
H, inliers = cv2.findHomography(pixels, world, cv2.RANSAC, 0.005)   # 5 mm RANSAC threshold

# Reprojection error: how far the mapped corners land from their measured robot position
mapped = cv2.perspectiveTransform(pixels.reshape(-1, 1, 2), H).reshape(-1, 2)
errors = np.linalg.norm(mapped - world, axis=1) * 1000
print(f'{len(pixels)} corners from {len(pixels) // (BOARD_INNER[0] * BOARD_INNER[1])} images, '
      f'{int(inliers.sum())} inliers')
print(f'Reprojection error: mean {errors.mean():.2f} mm, max {errors.max():.2f} mm')

np.save('homography.npy', H)
print('Saved homography.npy\n', H)
first = sorted(glob.glob('calib_*.jpg'))[0]
show(draw_board(load_image(first), find_board(load_image(first))), f'{first} — corners A, B, C')

A mean error of a few millimetres is good. A small error only means that the corners are consistent with **each other**. It does not prove that A, B and C were measured on the right corners, so check the labels in the image above against the corners you measured.

## 🧩 Exercise 6 — Pixel to Robot Coordinate Mapping
`cv2.perspectiveTransform` applies $H$ and the division by the third coordinate for us.

In [ ]:
# Student Task:
H = np.load('homography.npy')

def pixel_to_robot(px, py, H=H):
    """Pixel (px, py) -> robot (X, Y) in metres, on the table plane."""
    X, Y = cv2.perspectiveTransform(np.array([[[px, py]]], dtype=np.float64), H)[0, 0]
    return float(X), float(Y)

# Sanity check: what part of the table does the camera see? It must overlap the robot workspace.
h, w = load_image('snapshot.jpg').shape[:2]
for name, (u, v) in {'top-left': (0, 0), 'top-right': (w - 1, 0),
                     'bottom-right': (w - 1, h - 1), 'bottom-left': (0, h - 1)}.items():
    X, Y = pixel_to_robot(u, v)
    print(f'image {name:12s} -> robot X={X:+.3f} m  Y={Y:+.3f} m')
print()

# Objects from Exercise 3, in robot coordinates
for o in detect_colored_objects(load_image('snapshot.jpg')):
    X, Y = pixel_to_robot(*o['center'])
    print(f"{o['color']:6s} pixel ({o['center'][0]:6.1f}, {o['center'][1]:6.1f}) -> robot X={X:+.3f} m  Y={Y:+.3f} m")

## 🎓 Final Challenge — Vision-Guided Pick Simulation
Close the loop **perception → decision → action**:
1. Detect colored objects (Exercise 3).
2. Compute their pixel center.
3. Convert to robot coordinates (Exercise 6).
4. Move the TCP above the object in URSim, go down, **close the gripper**, lift, and drop the object in the bin for its color.

URSim has no gripper attached, so the gripper is simulated by **tool digital output 0** (`True` = closed). Watch it change in Polyscope under *I/O*.

The cell below connects to URSim and gives you `safe_move_l()` (Cartesian moves with safety checks, like Day 1's `safe_move`) and `gripper()`.

In [ ]:
# Connect to URSim. Running this cell again first closes the previous connection,
# which avoids "RTDE input registers are already in use".
from rtde_control import RTDEControlInterface
from rtde_receive import RTDEReceiveInterface
from rtde_io import RTDEIOInterface

ROBOT_IP = '127.0.0.1'
for name in ('rtde_c', 'rtde_io', 'rtde_r'):
    if name in globals():
        try:
            if name == 'rtde_c':
                rtde_c.stopScript()
            globals()[name].disconnect()
        except Exception:
            pass
rtde_c = RTDEControlInterface(ROBOT_IP)
rtde_r = RTDEReceiveInterface(ROBOT_IP)
rtde_io = RTDEIOInterface(ROBOT_IP)

TOOL_DOWN = [0, math.pi, 0]           # rotation vector: tool Z axis pointing down at the table
WORKSPACE = {'x': (-0.80, -0.15), 'y': (-0.60, 0.60), 'z': (0.01, 0.60)}   # m, allowed TCP box
MAX_LIN_SPEED = 0.25                  # m/s
MAX_LIN_ACCEL = 0.5                   # m/s²
POSITION_TOLERANCE = 0.002            # m
GRIPPER_DO = 0                        # tool digital output used as the gripper

def safe_move_l(pose, speed=0.1, accel=0.3):
    """Linear move to pose [x, y, z, rx, ry, rz] only if all checks pass. Returns True on success."""
    if len(pose) != 6 or not all(math.isfinite(v) for v in pose):
        print('❌ pose must be 6 finite numbers'); return False
    if not (0 < speed <= MAX_LIN_SPEED and 0 < accel <= MAX_LIN_ACCEL):
        print(f'❌ speed/accel out of range: {speed}, {accel}'); return False
    if rtde_r.isProtectiveStopped() or rtde_r.isEmergencyStopped() \
            or rtde_r.getRobotMode() != 7 or rtde_r.getSafetyMode() != 1:
        print('❌ robot not ready'); return False
    for value, axis in zip(pose[:3], 'xyz'):
        low, high = WORKSPACE[axis]
        if not low <= value <= high:
            print(f'❌ {axis}={value:.3f} m outside workspace [{low}, {high}]'); return False
    if not rtde_c.isPoseWithinSafetyLimits(pose):
        print('❌ pose violates the controller safety limits'); return False
    if not rtde_c.moveL(pose, speed, accel):
        print('❌ moveL failed'); return False
    error = np.linalg.norm(np.array(rtde_r.getActualTCPPose()[:3]) - np.array(pose[:3]))
    if error > POSITION_TOLERANCE:
        print(f'❌ position error {error * 1000:.1f} mm'); return False
    return True

def gripper(close):
    """Close (True) or open (False) the simulated gripper and wait for it."""
    rtde_io.setToolDigitalOut(GRIPPER_DO, close)
    time.sleep(0.5)
    return rtde_r.getDigitalOutState(16 + GRIPPER_DO) == close   # tool outputs are bits 16-17

print('Robot mode:', rtde_r.getRobotMode(), '| TCP:', np.round(rtde_r.getActualTCPPose(), 3))

In [ ]:
APPROACH_Z = 0.15      # m, height above the table when travelling
PICK_Z = 0.02          # m, TCP height when grasping
DROP_ZONES = {         # robot (X, Y) of the bin for each color
    'red':    (-0.30, 0.35),
    'yellow': (-0.40, 0.35),
    'green':  (-0.50, 0.35),
    'blue':   (-0.60, 0.35),
}

def pick_and_place(obj):
    """Pick one detected object and drop it in the bin for its color. Returns True on success."""
    X, Y = pixel_to_robot(*obj['center'])
    bin_x, bin_y = DROP_ZONES[obj['color']]
    print(f"→ {obj['color']} object at X={X:+.3f}, Y={Y:+.3f} m")
    steps = [
        ('open gripper',      lambda: gripper(False)),
        ('above object',      lambda: safe_move_l([X, Y, APPROACH_Z] + TOOL_DOWN)),
        ('down to object',    lambda: safe_move_l([X, Y, PICK_Z] + TOOL_DOWN, speed=0.05)),
        ('close gripper',     lambda: gripper(True)),
        ('lift',              lambda: safe_move_l([X, Y, APPROACH_Z] + TOOL_DOWN, speed=0.05)),
        ('above bin',         lambda: safe_move_l([bin_x, bin_y, APPROACH_Z] + TOOL_DOWN)),
        ('release',           lambda: gripper(False)),
    ]
    for name, step in steps:
        if not step():
            print(f'   ❌ step "{name}" failed, stopping')
            return False
        print(f'   ✅ {name}')
    return True

# 1. Perception: one fresh frame (press q once the objects are in place)
frame = run_live('Pick — press q to start')
if frame is None:
    raise RuntimeError('No camera frame')
objects = [o for o in detect_colored_objects(frame) if o['color'] in DROP_ZONES]
show(draw_objects(frame.copy(), objects), f'{len(objects)} objects to pick')

# 2. Decision + action: pick them one by one, stop at the first failure
for obj in objects:
    if not pick_and_place(obj):
        break
safe_move_l([-0.45, 0.0, 0.30] + TOOL_DOWN)   # park position

## Cleanup
Release the robot so another notebook or script can take control.

In [ ]:
rtde_c.stopScript()
rtde_c.disconnect()
rtde_io.disconnect()
rtde_r.disconnect()